## Easy Round 3

**Design a simple internal notification service.**

Employees should be able to receive notifications when:

* a build fails
* a ticket is assigned to them
* a deployment completes

They should be able to:

* view recent notifications
* mark them as read

Keep it mini interview style:

* requirements
* API
* data model
* scale
* failure cases
* security
* observability

Start with clarification questions or jump into your design.


To design an effective internal notification service, I will first outline a few **clarification questions** to narrow the scope:
*   **Channels:** Should we support multiple delivery methods like email and SMS, or just an internal UI dashboard?
*   **Scale:** What is the expected number of employees (Daily Active Users) and peak notification volume?
*   **Latency:** Is "soft real-time" (slight delay acceptable under load) sufficient?
*   **Retention:** How long should we keep notification history before it is archived or deleted?

Assuming a mid-to-large company with a "soft real-time" requirement, here is the design:

### **1. Requirements**
*   **Functional:** Support triggers for **build failures**, **ticket assignments**, and **deployment completions**; allow users to **view** and **mark notifications as read** [User prompt].
*   **Non-Functional:**
    *   **Reliability:** Ensure **no data loss**; notifications must be persisted and retried if delivery fails.
    *   **Scalability:** Must handle spikes in traffic (e.g., many builds failing simultaneously).
    *   **Extensibility:** Easily add new notification types or channels in the future.

### **2. API Sketch**
*   **POST** `/v1/notifications/trigger`
    *   **Internal use only:** Called by the CI/CD system or ticketing tool.
    *   **Input:** `{ "user_id": "string", "event_type": "build_fail", "metadata": {...} }`.
*   **GET** `/v1/notifications`
    *   **Output:** Returns a JSON list of the **most recent notifications** for the authenticated employee.
*   **PATCH** `/v1/notifications/{id}/read`
    *   **Action:** Updates the **read status** of a specific notification.

### **3. Data Model**
A **relational database** (e.g., PostgreSQL) is ideal for maintaining the consistency of notification states.
*   **User Table:** `user_id`, `email`, `settings` (e.g., opt-in preferences).
*   **Notification Table:** `id`, `user_id`, `type` (build/ticket/deploy), `content`, `is_read` (boolean), and `created_at` (timestamp).
*   **Indexes:** A **concatenated index** on `(user_id, created_at)` allows the UI to quickly fetch the most recent alerts for a specific employee.

### **4. Scale Plan**
*   **Message Queues:** Use a system like **Kafka or RabbitMQ** to decouple the triggering services (CI/CD) from the delivery logic.
*   **Stateless Web Tier:** API servers should be stateless to allow **horizontal scaling** by adding more servers during peak deployment windows.
*   **Caching:** Use an in-memory **Cache Tier** (e.g., Redis) to store the latest unread notifications for each user, reducing the read load on the primary database.

### **5. Failure Cases**
*   **Retry Mechanism:** If the database or a third-party channel (like email) is down, workers will **put the event back into the queue** for a predefined number of retries.
*   **Deduplication:** To avoid sending the same notification twice due to network retries, use a **dedupe logic** that checks a unique `event_id` before processing.
*   **Backpressure:** If the system is overwhelmed, the **message queue acts as a buffer** to prevent the database from crashing.

### **6. Security**
*   **Authentication:** Integrate with the company's **Single Sign-On (SSO)**; only verified internal clients can trigger the `send` API.
*   **Input Validation:** Sanitize all incoming notification content to prevent **malicious scripts** or formatting errors.
*   **Least Privilege:** Ensure the notification workers only have **read/write access** to necessary notification tables, not the entire employee database.

### **7. Observability**
*   **SLIs/SLOs:** Monitor the **99th percentile delivery latency** and **success rate**.
*   **Metrics:** Track **queue depth**; a large number of queued messages indicates that more workers are needed to handle the load.
*   **Centralized Logging:** Log all state changes (e.g., when a notification is "sent" vs "delivered") for **auditing and debugging** during failures.